## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None
!pip install --upgrade lightgbm >> None
!pip install --upgrade xgboost >> None

## 2. Импорт библиотек и настройка окружения

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from catboost import CatBoostRegressor, Pool
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Предобработка данных

In [ ]:
target = df_target[['client_num', 'target']]

data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Данные разделены на обучающую и валидационную выборки.")

## 5. Расчет весов классов

In [ ]:
unique_classes = np.sort(y.unique())
class_counts = y_train.value_counts()
total_samples = len(y_train)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        target = np.array(target)
        weight = np.ones_like(target) if weight is None else np.array(weight)
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
print("Кастомная метрика WMAE определена.")

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для выборок рассчитаны.")

## 6. Предобработка данных для моделей

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ],
    remainder='passthrough'
)

results = {}

num_features = [col for col in X.columns if col not in cat_features]

## 7. Предобработка данных для XGBoost

In [ ]:
print("\nОбучение модели XGBoost...")

for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_cat = oe.fit_transform(X_train[cat_features])
X_valid_cat = oe.transform(X_valid[cat_features])

X_train_encoded = X_train.copy()
X_valid_encoded = X_valid.copy()

X_train_encoded[cat_features] = X_train_cat
X_valid_encoded[cat_features] = X_valid_cat

X_train_encoded[num_features] = X_train_encoded[num_features].astype(float)
X_valid_encoded[num_features] = X_valid_encoded[num_features].astype(float)

X_train_encoded[cat_features] = X_train_encoded[cat_features].astype(int)
X_valid_encoded[cat_features] = X_valid_encoded[cat_features].astype(int)

feature_names = X_train_encoded.columns.tolist()

feature_types = []
for col in feature_names:
    if col in cat_features:
        feature_types.append('c')
    else:
        feature_types.append('q')

## 8. Обучение модели XGBoost

In [ ]:
dtrain = xgb.DMatrix(
    data=X_train_encoded, 
    label=y_train, 
    weight=weights_train,
    feature_names=feature_names,
    feature_types=feature_types
)

dvalid = xgb.DMatrix(
    data=X_valid_encoded, 
    label=y_valid, 
    weight=weights_valid,
    feature_names=feature_names,
    feature_types=feature_types
)

params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,      
    'max_depth': 11,              
    'lambda': 2.0,               
    'alpha': 0.5,                
    'subsample': 0.8,           
    'colsample_bytree': 0.8,     
    'tree_method': 'hist',       
    'eval_metric': 'mae',        
    'enable_categorical': True,
    'seed': 42                   
}

evals = [(dtrain, 'train'), (dvalid, 'eval')]

bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=100_000,           
    evals=evals,
    early_stopping_rounds=50,       
    verbose_eval=10                 
)

y_valid_pred = bst.predict(dvalid)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('XGBoost Validation WMAE:', wmae)

results['XGBoost'] = wmae

## 9. Предобработка данных для LightGBM

In [ ]:
print("\nОбучение модели LightGBM...")

for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')

categorical_feature_indices = [X_train.columns.get_loc(col) for col in cat_features]

## 10. Обучение модели LightGBM

In [ ]:
lgb_train = lgb.Dataset(
    data=X_train,
    label=y_train,
    weight=weights_train,
    categorical_feature=categorical_feature_indices
)

lgb_valid = lgb.Dataset(
    data=X_valid,
    label=y_valid,
    weight=weights_valid,
    reference=lgb_train,
    categorical_feature=categorical_feature_indices
)

params = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,         
    'max_depth': 11,               
    'lambda_l1': 0.5,               
    'lambda_l2': 2.0,               
    'feature_fraction': 0.4,        
    'bagging_fraction': 0.8,        
    'bagging_freq': 5,              
    'min_data_in_leaf': 20,   
    'random_state': 42,
    'verbose': 10                   
}

num_round = 100_000                   
early_stopping_rounds = 100

callbacks = [
    lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=True),
    lgb.log_evaluation(period=50)
]

bst = lgb.train(
    params=params,
    train_set=lgb_train,
    valid_sets=[lgb_valid],
    num_boost_round=num_round,
    callbacks=callbacks
)

y_valid_pred = bst.predict(X_valid, num_iteration=bst.best_iteration)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('LightGBM Validation WMAE:', wmae)

results = {}
results['LightGBM'] = wmae

## 11. Обучение модели RandomForestRegressor

In [ ]:
print("\nОбучение модели RandomForestRegressor...")

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        max_depth=11,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(
    X_train, y_train,
    regressor__sample_weight=weights_train
)

y_valid_pred = rf_model.predict(X_valid)
wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('RandomForestRegressor Validation WMAE:', wmae)
results['RandomForestRegressor'] = wmae

## 12. Обучение модели LinearRegression

In [ ]:
print("\nОбучение модели LinearRegression...")
lr_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

lr_model.fit(
    X_train, y_train,
    regressor__sample_weight=weights_train
)

y_valid_pred = lr_model.predict(X_valid)
wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('LinearRegression Validation WMAE:', wmae)
results['LinearRegression'] = wmae

## 13. Предобработка данных для CatBoost

In [ ]:
print("\nОбучение модели CatBoost...")

for col in cat_features:
    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)

train_pool = Pool(
    data=X_train, 
    label=y_train, 
    weight=weights_train, 
    cat_features=cat_features
)
valid_pool = Pool(
    data=X_valid, 
    label=y_valid, 
    weight=weights_valid, 
    cat_features=cat_features
)
print("Данные подготовлены для CatBoost.")

## 14. Обучение модели CatBoost

In [ ]:
params = {
    'iterations': 1_000_000,
    'depth': 11,
    'l2_leaf_reg': 2,
    'colsample_bylevel': 0.4,
    'boosting_type': 'Plain',
    'bootstrap_type': 'MVS',
    'eval_metric': WMAEMetric(),
    'loss_function': 'MAE',
    'random_seed': 42,
    'verbose': True,
    'early_stopping_rounds': 300,
    'use_best_model': True
}
print("Параметры модели настроены.")

model = CatBoostRegressor(**params)

model.fit(
    train_pool,
    eval_set=valid_pool,
    verbose=True
)
print("Модель обучена.")

y_valid_pred = model.predict(X_valid)

wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('CatBoost Validation WMAE:', wmae)

results['CatBoost'] = wmae

## 15. Результаты работы

In [ ]:
results_df = pd.DataFrame({
    'Название модели': list(results.keys()),
    'WMAE metric': list(results.values())
})

print("\nРезультаты:")
print(results_df)